# Capstone 02 – Custom Knowledge Chatbot (Excel + Text Based)
## RAG Pipeline Notebook — Atif Freelance Web Dev

This notebook demonstrates a complete **Retrieval-Augmented Generation (RAG)** pipeline that:
1. Ingests Excel and Text files
2. Converts content to TF-IDF embeddings
3. Stores in a local vector-style index
4. Retrieves relevant context for user queries
5. Generates answers using a LLM (Gemini API)

---


## 1. Install Dependencies

In [ ]:
# Run this cell once
# !pip install pandas openpyxl scikit-learn chromadb google-generativeai streamlit
print("Dependencies ready")


## 2. Import Libraries

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("All imports successful ✓")


## 3. Configuration
Set your file paths and API key here.


In [ ]:
# ── Configuration ────────────────────────────────────────────────────
GEMINI_API_KEY = ""   # Paste your Google Gemini API key here
                       # Get free key at: https://aistudio.google.com/

DATA_DIR       = "./data"          # Folder with your Excel / text files
INDEX_PATH     = "./tfidf_index.pkl"  # Where to save the index
TOP_K          = 5                 # How many chunks to retrieve per query

print("Config set. DATA_DIR:", DATA_DIR)


## 4. Text Chunking
Large documents are split into smaller overlapping chunks so we can find the most relevant pieces.

In [ ]:
def chunk_text(text, chunk_size=200, overlap=30):
    """Split text into overlapping word-based chunks."""
    words  = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i : i + chunk_size])
        chunks.append(chunk.strip())
        i += chunk_size - overlap
    return [c for c in chunks if len(c) > 20]

# Demo
sample = "Hello this is a test. " * 50
chunks = chunk_text(sample, chunk_size=20, overlap=5)
print(f"Original: {len(sample.split())} words")
print(f"Chunks:   {len(chunks)}")
print(f"Sample:   {chunks[0][:60]}...")


## 5. File Parsers
Supports `.txt`, `.md` and `.xlsx` / `.csv` files.

In [ ]:
def parse_txt(file_path):
    """Read a text or markdown file, return list of (chunk, metadata) tuples."""
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    filename = os.path.basename(file_path)
    chunks   = chunk_text(content)
    return [(c, {"source": filename, "type": "text"}) for c in chunks]


def parse_excel(file_path):
    """Read all sheets in an Excel file. Each row becomes one document."""
    xl       = pd.ExcelFile(file_path)
    docs     = []
    filename = os.path.basename(file_path)
    for sheet in xl.sheet_names:
        df = xl.parse(sheet).fillna("")
        for _, row in df.iterrows():
            row_text = f"[Sheet: {sheet}] " + " | ".join(
                f"{col}: {val}" for col, val in row.items() if str(val).strip()
            )
            docs.append((row_text, {"source": filename, "sheet": sheet, "type": "excel"}))
    return docs


print("Parsers defined ✓")


## 6. Load and Parse the Sample Data

In [ ]:
# Parse sample files
txt_docs   = parse_txt(os.path.join(DATA_DIR, "company_faq.txt"))
excel_docs = parse_excel(os.path.join(DATA_DIR, "store_data.xlsx"))

all_docs   = txt_docs + excel_docs

print(f"Text file chunks  : {len(txt_docs)}")
print(f"Excel rows        : {len(excel_docs)}")
print(f"Total documents   : {len(all_docs)}")
print()
print("Sample document:")
print(all_docs[0][0][:200])
print("Metadata:", all_docs[0][1])


## 7. Build TF-IDF Index (The "Embeddings")

**What is TF-IDF?**
- **TF** (Term Frequency) = how often a word appears in a document
- **IDF** (Inverse Document Frequency) = how rare the word is across all documents
- Multiplied together, TF-IDF gives a score showing how *important* a word is to a document

This creates a mathematical vector for each chunk, allowing us to measure similarity between a query and all chunks.

> 💡 In production RAG systems, you'd use neural embeddings (like `sentence-transformers`) instead of TF-IDF — they understand meaning, not just word frequency. The RAG *concept* is identical.


In [ ]:
def build_index(documents):
    """
    Build a TF-IDF index from (text, metadata) tuples.
    Returns: vectorizer, matrix, doc_store
    """
    texts     = [d[0] for d in documents]
    metadatas = [d[1] for d in documents]

    print(f"Building TF-IDF index for {len(texts)} documents...")
    vectorizer = TfidfVectorizer(
        ngram_range  = (1, 2),   # unigrams + bigrams
        max_features = 10000,    # top 10k terms
        sublinear_tf = True      # dampen high-frequency terms
    )
    matrix    = vectorizer.fit_transform(texts)
    doc_store = [{"text": t, "metadata": m} for t, m in zip(texts, metadatas)]

    print(f"Index shape: {matrix.shape}  (documents × terms)")
    return vectorizer, matrix, doc_store

vectorizer, tfidf_matrix, doc_store = build_index(all_docs)


## 8. Save the Index to Disk

In [ ]:
def save_index(vectorizer, matrix, doc_store, path=INDEX_PATH):
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump({"vectorizer": vectorizer, "matrix": matrix, "docs": doc_store}, f)
    size_kb = os.path.getsize(path) / 1024
    print(f"Index saved to {path}  ({size_kb:.1f} KB)")

def load_index(path=INDEX_PATH):
    with open(path, "rb") as f:
        obj = pickle.load(f)
    print(f"Index loaded: {len(obj['docs'])} documents")
    return obj["vectorizer"], obj["matrix"], obj["docs"]

save_index(vectorizer, tfidf_matrix, doc_store)


## 9. Retrieval Function
For a given query, we find the most similar chunks using cosine similarity.

In [ ]:
def retrieve(query, vectorizer, matrix, doc_store, top_k=TOP_K):
    """
    Find the top-K most relevant chunks for a query.
    
    How it works:
    1. Convert query to TF-IDF vector using the same vocabulary
    2. Compute cosine similarity between query and ALL document vectors
    3. Return top-K highest similarity chunks
    """
    q_vec  = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, matrix).flatten()
    top_idx = scores.argsort()[::-1][:top_k]

    results = []
    for i in top_idx:
        if scores[i] > 0:
            doc = doc_store[i]
            results.append({
                "text":   doc["text"],
                "source": doc["metadata"].get("source", "?"),
                "sheet":  doc["metadata"].get("sheet", ""),
                "score":  round(float(scores[i]), 4)
            })
    return results

print("Retrieval function defined ✓")


## 10. Test Retrieval — Does It Find the Right Chunks?

In [ ]:
test_queries = [
    "What is the return policy?",
    "How much does the Dell laptop cost?",
    "What payment methods are accepted?",
    "Who is the customer from Lahore?",
    "Tell me about Sony headphones",
]

for q in test_queries:
    results = retrieve(q, vectorizer, tfidf_matrix, doc_store, top_k=2)
    print(f"Query : {q}")
    for r in results:
        src = f"{r['source']}" + (f"/{r['sheet']}" if r['sheet'] else "")
        print(f"  [{r['score']}] {src}")
        print(f"         {r['text'][:100]}...")
    print()


## 11. Build the RAG Prompt

In [ ]:
def build_prompt(query, chunks):
    """Combine retrieved context with user question into a prompt for the LLM."""
    context_parts = []
    for c in chunks:
        src = c['source'] + (f"/{c['sheet']}" if c.get('sheet') else "")
        context_parts.append(f"[Source: {src}]\n{c['text']}")
    context = "\n\n".join(context_parts)

    prompt = f"""You are a helpful assistant. Answer the user's question using ONLY the context below.
If the answer is not in the context, say "I don't have that information in the knowledge base."

Context:
{context}

Question: {query}

Answer:"""
    return prompt

# Show example prompt
q      = "What is the return policy?"
chunks = retrieve(q, vectorizer, tfidf_matrix, doc_store, top_k=3)
prompt = build_prompt(q, chunks)
print(prompt[:600])
print("...")


## 12. Call the LLM (Gemini API)

Set your `GEMINI_API_KEY` in Cell 3 to get real answers.
Get a **free** key at: https://aistudio.google.com/

If no key is set, the cell shows the raw retrieved context instead.


In [ ]:
def call_gemini(prompt, api_key):
    import urllib.request
    url  = (
        "https://generativelanguage.googleapis.com/v1beta/"
        f"models/gemini-2.0-flash:generateContent?key={api_key}"
    )
    body = json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode()
    req  = urllib.request.Request(
        url, data=body, headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read())
    return data["candidates"][0]["content"]["parts"][0]["text"]


def rag_answer(query, api_key=None):
    """Full RAG pipeline: retrieve → build prompt → call LLM → return answer."""
    chunks = retrieve(query, vectorizer, tfidf_matrix, doc_store)
    if not chunks:
        return "No relevant information found.", []

    prompt = build_prompt(query, chunks)

    if api_key:
        answer_text = call_gemini(prompt, api_key)
    else:
        # Demo mode: show raw context
        answer_text = "[No API key — showing raw context]\n\n"
        answer_text += "\n\n---\n".join(
            f"Source: {c['source']}\n{c['text'][:250]}" for c in chunks[:2]
        )
    return answer_text, chunks


# ── Run a test query ──────────────────────────────────────────────────────────
answer_text, sources = rag_answer("What is the return policy?", api_key=GEMINI_API_KEY or None)
print("Answer:")
print(answer_text)
print()
print("Sources used:")
for s in sources:
    print(f"  - {s['source']} (score: {s['score']})")


## 13. Multi-Turn Conversation (Chat History)

In [ ]:
def chat_session(api_key=None):
    """
    Simple interactive chat session.
    Each question is answered independently using RAG.
    Type 'quit' to exit.
    """
    print("=== RAG Chatbot Ready ===")
    print("Type your question and press Enter. Type 'quit' to stop.\n")
    history = []

    while True:
        user_input = input("You: ").strip()
        if not user_input or user_input.lower() in ("quit", "exit"):
            print("Goodbye!")
            break

        answer_text, sources = rag_answer(user_input, api_key=api_key)
        print(f"Bot: {answer_text}")
        print(f"     [Sources: {', '.join(set(s['source'] for s in sources))}]\n")
        history.append({"user": user_input, "bot": answer_text})

    return history

# Uncomment to run interactive session:
# history = chat_session(api_key=GEMINI_API_KEY or None)
print("chat_session() defined. Uncomment last line to run interactively.")


## 14. Evaluation — Testing Retrieval Quality

In [ ]:
# Quick evaluation: check if correct source is retrieved for known questions
eval_set = [
    ("What is the return policy?",        "company_faq.txt"),
    ("How much does the Dell laptop cost?", "store_data.xlsx"),
    ("Who is the customer from Lahore?",   "store_data.xlsx"),
    ("What payment methods do you accept?","company_faq.txt"),
    ("Tell me about Sony headphones",      "store_data.xlsx"),
]

correct = 0
print(f"{'Query':<45} {'Expected':<20} {'Got':<20} {'Pass'}")
print("-" * 95)
for query, expected_src in eval_set:
    results = retrieve(query, vectorizer, tfidf_matrix, doc_store, top_k=1)
    got_src  = results[0]["source"] if results else "none"
    passed   = expected_src in got_src
    correct += int(passed)
    print(f"{query:<45} {expected_src:<20} {got_src:<20} {'✓' if passed else '✗'}")

print(f"\nAccuracy: {correct}/{len(eval_set)} = {correct/len(eval_set)*100:.0f}%")


## 15. Summary

### What you built:
| Component | Technology | Purpose |
|---|---|---|
| File parser | `pandas`, built-in | Read Excel + text files |
| Chunker | Custom Python | Split docs into searchable pieces |
| Embedder | `TF-IDF (sklearn)` | Convert text to vectors |
| Vector store | `pickle` file | Store and retrieve vectors |
| Retriever | Cosine similarity | Find most relevant chunks |
| LLM | Gemini API (free) | Generate natural language answers |
| Frontend | Streamlit | User interface |

### Next steps to improve:
1. **Better embeddings** → use `sentence-transformers` (needs internet) for semantic search
2. **Persistent vector DB** → use ChromaDB or FAISS for large datasets  
3. **Chat history** → pass previous messages to LLM for context-aware answers
4. **Streaming** → stream LLM tokens live for better UX
5. **Reranking** → use a cross-encoder to re-rank retrieved chunks

### Run the Streamlit app:
```bash
streamlit run app.py
```
